# P2 – Fugenläufe (Zweig `p2-fugen`)

Rechnet unbeaufsichtigt 13 Varianten × 5 Läufe (nur 7 Kanäle): Training, Segmentierung von wall1–wall4,
Metriken gegen die GT-Masken, Fugen- und Steinebene, unser Maßstab (allein / verschmolzen / zerfallen),
Ensembles und eine Zusammenfassung im Vergleich zum veröffentlichten Modell.

**So geht's**

1. Zelle 1 und 2 ausführen. In Zelle 2 nur die Pfade prüfen.
2. Zelle 3 ausführen: prüft alles, rechnet nichts.
3. Zelle 4 ausführen: **START**. Die Warteschlange läuft als eigener Prozess weiter, auch wenn Jupyter
   oder der Browser geschlossen wird. Die Zelle wartet auf den Vorabtest (etwa 20–30 Minuten) und meldet
   „bestanden“ oder den Fehler.
4. Zelle 5 jederzeit: Stand, Prognose, letzte Logzeilen. Zelle 6: Zusammenfassung.

Nach einem Absturz oder Neustart einfach wieder Zelle 1, 2 und 4 ausführen: Fertiges wird übersprungen.
Ausgaben: `experiments/p2_*` je Variante, `experiments/p2_summary/` (Status, Log, Fehler, Zusammenfassung).

## 1. Imports

In [ ]:
import os, sys, json, time, glob, shutil, subprocess, dataclasses
from datetime import datetime

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from amg_pipeline.config import RunConfig, make_run_id
from amg_pipeline import paths
from amg_pipeline import p2_queue

print("amg_pipeline aus:", os.path.join(REPO_ROOT, "amg_pipeline"))

## 2. Pfade (nur hier anpassen)

In [ ]:
BASE = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress"

ORTHO_DIR     = os.path.join(BASE, "snippets_orthomosaics")
NORMALMAP_DIR = os.path.join(BASE, "snippets_normalmaps")
MASK_DIR      = os.path.join(BASE, "snippets_masks")

TEST_ORTHO_DIR     = os.path.join(BASE, "Testing", "01_test-images")
TEST_NORMALMAP_DIR = os.path.join(BASE, "Testing", "02_test-normals")
TEST_MASK_DIR      = os.path.join(BASE, "Testing", "03_test-masks")
WALLS = ["wall1", "wall2", "wall3", "wall4"]

COCO_JSON = os.path.join(BASE, "2025-08-10.json")   # für die Polygon-Zählung
EXPERIMENTS_ROOT = os.path.join(REPO_ROOT, "experiments")

N_RUNS = 5          # Läufe je Variante
SMOKE_WALL = "wall3"  # kleinste Testmauer, für den Vorabtest

BASE_CONFIG = RunConfig(
    channels=7, run_number=1, experiment_name="p2", experiments_root=EXPERIMENTS_ROOT,
    ortho_dir=ORTHO_DIR, normalmap_dir=NORMALMAP_DIR, mask_dir=MASK_DIR,
    test_ortho_dir=TEST_ORTHO_DIR, test_normalmap_dir=TEST_NORMALMAP_DIR,
    test_mask_dir=TEST_MASK_DIR, walls=WALLS,
    seed=42, n_epochs=300, batch_size=16, lr=1e-4,
    roi_operation="closing", kernel_radius=45,
)
SUM_DIR = os.path.join(EXPERIMENTS_ROOT, p2_queue.SUMMARY)
print("Ausgaben:", EXPERIMENTS_ROOT)

## 3. Prüfen (rechnet nichts)

In [ ]:
def _n(d):
    return len(glob.glob(os.path.join(d, "*.png"))) if os.path.isdir(d) else 0

ok = True
print("=== Trainingskacheln ===")
for label, d in (("ortho", ORTHO_DIR), ("normalmap", NORMALMAP_DIR), ("mask", MASK_DIR)):
    n = _n(d); ok &= n > 0
    print(f"  {'OK' if n else '!!'} {label:10s} {n:6d}  {d}")

print("\n=== Testmauern ===")
for w in WALLS:
    for label, fn in (("ortho", paths.test_ortho_path(BASE_CONFIG, w)),
                      ("normal", paths.test_normalmap_path(BASE_CONFIG, w)),
                      ("mask", paths.test_mask_path(BASE_CONFIG, w))):
        e = os.path.exists(fn); ok &= e
        print(f"  {'OK' if e else '!!'} {w} {label:6s} {fn}")

print("\n=== COCO-JSON (Polygone) ===")
print(f"  {'OK' if os.path.exists(COCO_JSON) else '(fehlt: nur Masken-Zählung)'} {COCO_JSON}")

print("\n=== Vorhersagen des Paper-Modells (Vergleich) ===")
for exp, rids in ((p2_queue.PAPER_RUNS, range(1, 6)), (p2_queue.PAPER_ENS, (1,))):
    for i in rids:
        c = dataclasses.replace(BASE_CONFIG, experiment_name=exp, run_number=i)
        have = sum(os.path.exists(paths.segmentation_raw_path(c, w)) for w in WALLS)
        print(f"  {'OK' if have == len(WALLS) else '!!'} {exp} 7ch_run{i}: {have}/{len(WALLS)} Raster")

free = shutil.disk_usage(EXPERIMENTS_ROOT).free / 1e9
print(f"\n=== Freier Platz: {free:.0f} GB ===")

print("\n=== Varianten (Reihenfolge je Runde) ===")
for name, ov in p2_queue.VARIANTS:
    done = sum(os.path.exists(paths.checkpoint_path(
        dataclasses.replace(BASE_CONFIG, experiment_name=name, run_number=r, **ov))) for r in range(1, N_RUNS + 1))
    print(f"  {name:20s} {done}/{N_RUNS} trainiert  {ov}")
print(f"\n{len(p2_queue.VARIANTS)} Varianten x {N_RUNS} Läufe = {len(p2_queue.VARIANTS) * N_RUNS} Läufe; "
      f"geschätzt ~2 h Training je Lauf -> grob {len(p2_queue.VARIANTS) * N_RUNS * 2 / 24:.1f} Tage plus Auswertung.")
print("\nAlles da." if ok else "\n*** Etwas fehlt (siehe !!) – vor dem Start beheben. ***")

## 4. START

Startet die Warteschlange als eigenen Prozess und wartet auf den Vorabtest. Erst wenn hier
**„Vorabtest bestanden“** steht, laufen die langen Läufe.

In [ ]:
os.makedirs(SUM_DIR, exist_ok=True)
hb_path = os.path.join(SUM_DIR, "heartbeat.txt")
status_path = os.path.join(SUM_DIR, "status.json")

def heartbeat_age():
    try:
        return time.time() - float(open(hb_path).read().split()[0])
    except Exception:
        return None

age = heartbeat_age()
if age is not None and age < 180:
    print(f"Die Warteschlange läuft bereits (Lebenszeichen vor {age:.0f} s). Nicht doppelt starten – Stand in Zelle 5.")
else:
    qcfg = {"base": dataclasses.asdict(BASE_CONFIG), "n_runs": N_RUNS, "coco_json": COCO_JSON,
            "smoke_wall": SMOKE_WALL, "smoke_tiles": 32}
    cfg_path = os.path.join(SUM_DIR, "queue_config.json")
    with open(cfg_path, "w", encoding="utf-8") as f:
        json.dump(qcfg, f, indent=2, default=list)
    log_f = open(os.path.join(SUM_DIR, "queue_log.txt"), "a", encoding="utf-8")
    log_f.write(f"\n\n######## START {datetime.now():%Y-%m-%d %H:%M:%S} ########\n"); log_f.flush()
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    flags = (subprocess.CREATE_NEW_PROCESS_GROUP | 0x00000008) if os.name == "nt" else 0  # 0x8 = DETACHED_PROCESS
    proc = subprocess.Popen([sys.executable, "-m", "amg_pipeline.p2_queue", cfg_path], cwd=REPO_ROOT,
                            stdout=log_f, stderr=subprocess.STDOUT, env=env, creationflags=flags)
    print(f"Gestartet (PID {proc.pid}). Warte auf den Vorabtest ...")
    last, t0 = None, time.time()
    while True:
        time.sleep(15)
        try:
            st = json.load(open(status_path, encoding="utf-8"))
        except Exception:
            st = {}
        ph = st.get("phase")
        if ph != last:
            print(f"  {datetime.now():%H:%M:%S}  Phase: {ph}")
            last = ph
        if ph in ("4_main_queue", "4b_retry", "5_ensembles", "6_done"):
            print("\nVorabtest bestanden. Die langen Läufe laufen jetzt; du kannst den Rechner allein lassen.")
            print("Prüfung schnelles Closing:", st.get("morph"))
            print("Abgleich mit v9_gapstone (max. Abweichung):", st.get("reference_check"))
            print("Vorabtest je Variante (s):", st.get("smoke_seconds"))
            break
        if ph in ("smoke_failed", "checks_failed") or proc.poll() is not None:
            print(f"\n*** Abgebrochen in Phase {ph} (Exit {proc.poll()}). Ursache unten und in p2_summary/errors.log ***")
            print("Probleme:", st.get("problems"), "| Vorabtest:", st.get("smoke"))
            print("".join(open(os.path.join(SUM_DIR, "queue_log.txt"), encoding="utf-8", errors="replace").readlines()[-40:]))
            break
        if time.time() - t0 > 2.5 * 3600:
            print("Vorabtest dauert ungewöhnlich lange – Stand in Zelle 5 ansehen.")
            break

## 5. Stand (jederzeit ausführen)

In [ ]:
try:
    st = json.load(open(os.path.join(SUM_DIR, "status.json"), encoding="utf-8"))
except FileNotFoundError:
    st = {}
try:
    age = time.time() - float(open(os.path.join(SUM_DIR, "heartbeat.txt")).read().split()[0])
except Exception:
    age = None
print(f"Lebenszeichen: {'vor %.0f s' % age if age is not None else 'keins'}"
      f"{'  -> läuft' if age is not None and age < 180 else '  -> läuft NICHT'}")
for k in ("phase", "current", "current_since", "items_done", "items_total", "items_failed",
          "mean_hours_per_run", "eta", "morph", "smoke", "last_start", "updated"):
    if k in st:
        print(f"  {k:18s} {st[k]}")
print("\n--- letzte Logzeilen ---")
lp = os.path.join(SUM_DIR, "queue_log.txt")
if os.path.exists(lp):
    print("".join(open(lp, encoding="utf-8", errors="replace").readlines()[-25:]))
ep = os.path.join(SUM_DIR, "errors.log")
if os.path.exists(ep):
    print("--- errors.log (Ende) ---")
    print("".join(open(ep, encoding="utf-8", errors="replace").readlines()[-30:]))

## 6. Zusammenfassung

In [ ]:
from IPython.display import Markdown, display
mdp = os.path.join(SUM_DIR, "p2_summary.md")
display(Markdown(open(mdp, encoding="utf-8").read()) if os.path.exists(mdp) else "Noch keine Zusammenfassung.")

## 7. Anhalten (nur im Notfall)

Legt die Datei `STOP` an; die Warteschlange hält nach dem laufenden Schritt an (ein Training läuft noch zu
Ende). Neu starten mit Zelle 4.

In [ ]:
DO_STOP = False
if DO_STOP:
    open(os.path.join(SUM_DIR, "STOP"), "w").close()
    print("STOP angelegt – die Warteschlange hält nach dem laufenden Schritt an.")